# Run → Question Number Map

**Date:** 2026-02-09  
**Purpose:** Map GitHub Actions workflow run numbers to Metaculus question numbers  
**Output:** CSV file in `products/`

In [1]:
import json, re, subprocess, csv
from pathlib import Path
from datetime import date
from typing import Dict, List, Optional

REPO = "D-Enns/metac-bot-template"
WORKFLOW = "dre_run_bot_on_tournament.yaml"
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
TEST_LIMIT = 200  # Set to None for all runs

print(f"Repo: {REPO}")
print(f"Limit: {TEST_LIMIT}")

Repo: D-Enns/metac-bot-template
Limit: 200


In [2]:
def strip_ansi_codes(text: str) -> str:
    return re.sub(r'\x1b\[[0-9;]*m', '', text)

def get_workflow_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[Dict]:
    print("Fetching workflow runs...")
    cmd = ["gh", "api", f"repos/{repo}/actions/workflows/{workflow}/runs", "--paginate"]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if result.returncode != 0:
        print(f"Failed: {result.stderr}")
        return []
    clean = strip_ansi_codes(result.stdout)

    # Parse concatenated JSON objects via brace counting
    json_objects = []
    current = ""
    count = 0
    for char in clean:
        current += char
        if char == '{':
            count += 1
        elif char == '}':
            count -= 1
            if count == 0 and current.strip():
                json_objects.append(current.strip())
                current = ""

    runs = []
    for obj_str in json_objects:
        data = json.loads(obj_str)
        if 'workflow_runs' in data:
            runs.extend(data['workflow_runs'])

    simplified = [{'id': r['id'], 'run_number': r['run_number']} for r in runs]
    simplified.sort(key=lambda x: x['run_number'], reverse=True)
    if limit:
        simplified = simplified[:limit]
    print(f"{len(simplified)} runs")
    return simplified

def download_run_log(repo: str, run_id: int) -> Optional[str]:
    cmd = ["gh", "run", "view", str(run_id), "--repo", repo, "--log"]
    try:
        result = subprocess.run(cmd, capture_output=True, timeout=120)
        if result.returncode != 0:
            return None
        try:
            log_text = result.stdout.decode('utf-8', errors='replace')
        except:
            log_text = result.stdout.decode('latin-1', errors='replace')
        return strip_ansi_codes(log_text)
    except Exception as e:
        print(f" Error: {e}")
        return None

print("Functions defined")

Functions defined


In [3]:
QUESTION_URL = re.compile(r'https://www\.metaculus\.com/questions/(\d+)')
FOUND_RESEARCH = re.compile(r'Found Research for URL')

runs = get_workflow_runs(REPO, WORKFLOW, limit=TEST_LIMIT)
results = []

for i, run in enumerate(runs, 1):
    rn = run['run_number']
    print(f"[{i}/{len(runs)}] Run #{rn}...", end='')
    log = download_run_log(REPO, run['id'])
    if not log:
        print(" failed")
        results.append((rn, ''))
        continue
    qnum = ''
    if FOUND_RESEARCH.search(log):
        m = QUESTION_URL.search(log)
        if m:
            qnum = m.group(1)
    results.append((rn, qnum))
    print(f" Q:{qnum}" if qnum else " -")

with_q = sum(1 for _, q in results if q)
print(f"\nDone: {len(results)} runs, {with_q} with questions")

Fetching workflow runs...
200 runs
 -/200] Run #1246...
 -/200] Run #1245...
 -/200] Run #1244...
 -/200] Run #1243...
 -/200] Run #1242...
 -/200] Run #1241...
 -/200] Run #1240...
 -/200] Run #1239...
 -/200] Run #1238...
 Q:42040 Run #1237...
 -1/200] Run #1236...
 -2/200] Run #1235...
 -3/200] Run #1234...
 -4/200] Run #1233...
 -5/200] Run #1232...
 -6/200] Run #1231...
 Q:42039 Run #1230...
 -8/200] Run #1229...
 Q:42077 Run #1228...
 -0/200] Run #1227...
 -1/200] Run #1226...
 -2/200] Run #1225...
 -3/200] Run #1224...
 -4/200] Run #1223...
 -5/200] Run #1222...
 Q:42038 Run #1221...
 -7/200] Run #1220...
 -8/200] Run #1219...
 -9/200] Run #1218...
 -0/200] Run #1217...
 -1/200] Run #1216...
 -2/200] Run #1215...
 -3/200] Run #1214...
 -4/200] Run #1213...
 -5/200] Run #1212...
 -6/200] Run #1211...
 -7/200] Run #1210...
 -8/200] Run #1209...
 -9/200] Run #1208...
 -0/200] Run #1207...
 -1/200] Run #1206...
 -2/200] Run #1205...
 -3/200] Run #1204...
 -4/200] Run #1203...
 -5/20

In [4]:
output_file = OUTPUT_DIR / f"Run_Question_Map_{date.today()}.csv"
results.sort(key=lambda x: x[0], reverse=True)

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['workflow_run_number', 'question_number'])
    writer.writerows(results)

print(f"Saved: {output_file.name}")
print(f"Rows: {len(results)}")

Saved: Run_Question_Map_2026-02-09.csv
Rows: 200


In [5]:
output_file

WindowsPath('C:/Users/Donni/projects/metac_bot_Spring_2026/products/Run_Question_Map_2026-02-09.csv')